# Thai ↔ Tai Tham ByT5 proof of concept
This notebook prepares audited data, fine-tunes ByT5-small, evaluates the locked test split, and saves complete run metadata.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/translation_model'  # change if needed


In [ ]:
!pip -q install -r "{PROJECT_DIR}/requirements-colab.txt"


In [ ]:
import os, json, platform, subprocess, sys, torch
SEED = 20260802
BASE_MODEL = 'google/byt5-small'
OUTPUT_DIR = f'{PROJECT_DIR}/output/lanna-byt5-poc'
EXTERNAL_APPROVED = f'{PROJECT_DIR}/data/raw/external/approved_records.jsonl'
os.makedirs(OUTPUT_DIR, exist_ok=True)
external_arg = f'--external-input \"{EXTERNAL_APPROVED}\"' if os.path.exists(EXTERNAL_APPROVED) else ''
!python "{PROJECT_DIR}/scripts/prepare_dataset.py" --input "{PROJECT_DIR}/data/raw/lanna_dict.json" --verified-overrides "{PROJECT_DIR}/data/raw/verified_overrides.json" {external_arg} --output-dir "{PROJECT_DIR}/data/processed" --seed {SEED}


In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments, set_seed
set_seed(SEED)
files = {s: f'{PROJECT_DIR}/data/processed/{s}.jsonl' for s in ['train','validation','test']}
dataset = load_dataset('json', data_files=files)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)
prefix = {'th_to_lanna': 'translate Thai to Tai Tham: ', 'lanna_to_th': 'translate Tai Tham to Thai: '}
def preprocess(batch):
    inputs = [prefix[d] + s for d, s in zip(batch['direction'], batch['source'])]
    encoded = tokenizer(inputs, max_length=192, truncation=True)
    encoded['labels'] = tokenizer(text_target=batch['target'], max_length=192, truncation=True)['input_ids']
    return encoded
tokenized = dataset.map(preprocess, batched=True, remove_columns=dataset['train'].column_names)


In [ ]:
import evaluate, numpy as np
chrf = evaluate.load('chrf')
sacrebleu = evaluate.load('sacrebleu')
def compute_metrics(result):
    predictions, labels = result
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    pred_text = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    label_text = tokenizer.batch_decode(labels, skip_special_tokens=True)
    return {'chrf': chrf.compute(predictions=pred_text, references=label_text)['score'], 'bleu': sacrebleu.compute(predictions=pred_text, references=[[x] for x in label_text])['score'], 'exact_match': 100 * sum(a == b for a,b in zip(pred_text,label_text)) / max(1,len(label_text))}
args = Seq2SeqTrainingArguments(output_dir=OUTPUT_DIR, eval_strategy='epoch', save_strategy='epoch', learning_rate=3e-4, per_device_train_batch_size=8, per_device_eval_batch_size=8, gradient_accumulation_steps=2, num_train_epochs=4, fp16=torch.cuda.is_available(), predict_with_generate=True, generation_max_length=192, load_best_model_at_end=True, metric_for_best_model='chrf', greater_is_better=True, save_total_limit=2, logging_steps=10, report_to='none', seed=SEED)
trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=tokenized['train'], eval_dataset=tokenized['validation'], processing_class=tokenizer, data_collator=DataCollatorForSeq2Seq(tokenizer, model=model), compute_metrics=compute_metrics)
train_result = trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)


In [ ]:
metrics = trainer.evaluate(tokenized['test'], metric_key_prefix='test')
run_report = {'base_model': BASE_MODEL, 'seed': SEED, 'python': sys.version, 'platform': platform.platform(), 'torch': torch.__version__, 'cuda_available': torch.cuda.is_available(), 'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, 'training_args': args.to_dict(), 'train_metrics': train_result.metrics, 'test_metrics': metrics}
with open(f'{OUTPUT_DIR}/training_report.json','w',encoding='utf-8') as f: json.dump(run_report,f,ensure_ascii=False,indent=2,default=str)
print(json.dumps(run_report,ensure_ascii=False,indent=2,default=str))
!cd "{PROJECT_DIR}/output" && zip -qr lanna-byt5-poc.zip lanna-byt5-poc
print(f'Download: {PROJECT_DIR}/output/lanna-byt5-poc.zip')
